# Capítulo 20 · Laboratório 20: Wyckoff
*Análise Técnica dos Mercados Financeiros*, 4ª edição, Flávio Lemos, CMT

Dados até **30/09/2026**. Na primeira execução, o notebook baixa os dados do Yahoo Finance na sua máquina e guarda uma cópia local (`dados_cap20.pkl`), que as execuções seguintes reaproveitam. O provedor corrige o histórico (proventos, desdobramentos), por isso seus números podem diferir um pouco dos do livro. O laboratório mostra quatro exemplos reais rotulados (RENT3, Bitcoin, NVIDIA e SUZB3), testa molas e falsos rompimentos, sozinhos e na sequência completa, o esforço sem resultado e a lei de causa e efeito, e confere os alvos do S&P 500 de 2009. **Nada aqui reproduz o Wyckoff Detector**: as regras são públicas e simplificadas, e os rótulos são a leitura do autor.

In [ ]:
!pip install -q yfinance

In [ ]:
import logging
import os
import numpy as np
import pandas as pd
import yfinance as yf

logging.getLogger("yfinance").setLevel(logging.CRITICAL)
FIM = "2026-10-01"                      # dados até 30/09/2026
CACHE = "dados_cap20.pkl"
pc = lambda x, k=0: f"{x:.{k}%}".replace(".", ",")
ps = lambda x, k=1: ("+" if x >= 0 else "") + pc(x, k)
nr = lambda x, k=2: f"{x:,.{k}f}".replace(",", "X").replace(".", ",").replace("X", ".")
B3 = ["PETR4", "VALE3", "ITUB4", "BBDC4", "BBAS3", "WEGE3", "ABEV3", "B3SA3", "RENT3", "SUZB3",
      "GGBR4", "PRIO3", "RADL3", "EQTL3", "VIVT3", "LREN3", "EMBJ3", "CSAN3", "RAIL3", "CMIG4"]
EUA = ["AAPL", "MSFT", "JPM", "JNJ", "PG", "KO", "XOM", "CVX", "WMT", "HD", "MRK", "PFE", "CSCO", "INTC",
       "IBM", "MMM", "CAT", "DIS", "MCD", "BA"]
TICKERS = [t + ".SA" for t in B3] + EUA
EXTRAS = ["BTC-USD", "NVDA", "^GSPC"]


def confere(baixados):
    """O Yahoo devolve uma tabela vazia quando falha: para aqui, com uma mensagem clara, sem gravar a cópia local."""
    for nome, x in baixados.items():
        if len(x) == 0:
            raise RuntimeError(f"Não consegui baixar {nome} do Yahoo Finance. O provedor pode estar fora do ar ou"
                               " limitando pedidos. Espere alguns minutos e rode a célula de novo.")


if os.path.exists(CACHE):
    D, DX = pd.read_pickle(CACHE)
else:
    D = yf.download(TICKERS + EXTRAS, start="2005-01-01", end=FIM, auto_adjust=True, progress=False,
                    group_by="ticker")
    DX = yf.download(["RENT3.SA", "BTC-USD", "NVDA", "SUZB3.SA"], start="2017-01-01", end=FIM, auto_adjust=False,
                     progress=False, group_by="ticker")              # exemplos: preços da época, como na plataforma
    confere({"as 40 ações, o Bitcoin, a NVIDIA e o S&P 500": D, "os exemplos rotulados": DX})
    pd.to_pickle((D, DX), CACHE)


def serie(t, fonte=None):
    d = (D if fonte is None else fonte)[t].drop(columns=["Adj Close"], errors="ignore").dropna()
    d = d[d["Close"] > 0]
    return d if t.startswith("^") else d[d["Volume"] > 0]

## Parte 1: dados

In [ ]:
print(f"Parte 1. {len(TICKERS)} ações (20 da B3 e 20 dos EUA) e {len(EXTRAS)} séries extras, de 2005 a"
      f" {serie('^GSPC').index[-1]:%d/%m/%Y}. Preços ajustados por proventos nos testes; nos exemplos,"
      f" preços da época (sem ajuste por proventos), como na plataforma.")

## Parte 2: quatro exemplos reais

In [ ]:
# Leitura do autor do capítulo, com datas fixas. Lado: "b" = evento no fundo da barra, "a" = no topo da barra.
EXEMPLOS = {
    "RENT3.SA": {"titulo": "RENT3: acumulação com mola (2022 e 2023)", "tipo": "acumulação",
                 "janela": ("2022-10-20", "2023-06-30"),
                 "eventos": [("PS", "2022-11-17", "b"), ("SC", "2022-12-14", "b"), ("AR", "2022-12-23", "a"),
                             ("ST", "2023-01-03", "b"), ("UT", "2023-02-02", "a"), ("Spring", "2023-03-15", "b"),
                             ("Teste", "2023-03-23", "b"), ("SOS", "2023-04-12", "a"), ("LPS", "2023-05-03", "b"),
                             ("JAC", "2023-05-11", "a"), ("BUEC", "2023-05-31", "b")],
                 "fases": [("A", "2022-11-17", "2023-01-03"), ("B", "2023-01-04", "2023-03-14"),
                           ("C", "2023-03-15", "2023-03-23"), ("D", "2023-03-24", "2023-05-11"),
                           ("E", "2023-05-12", "2023-06-30")],
                 "suporte": ("SC", "b"), "resistencia": ("UT", "a")},
    "BTC-USD": {"titulo": "Bitcoin: acumulação sem mola (2018 e 2019)", "tipo": "acumulação",
                "janela": ("2018-11-01", "2019-05-15"),
                "eventos": [("PS", "2018-11-25", "b"), ("SC", "2018-12-15", "b"), ("AR", "2018-12-24", "a"),
                            ("ST", "2019-01-29", "b"), ("Teste", "2019-02-07", "b"), ("SOS", "2019-02-18", "a"),
                            ("LPS", "2019-03-04", "b"), ("JAC", "2019-04-02", "a")],
                "fases": [("A", "2018-11-25", "2018-12-27"), ("B", "2018-12-28", "2019-02-06"),
                          ("C", "2019-02-07", "2019-02-08"), ("D", "2019-02-09", "2019-04-02"),
                          ("E", "2019-04-03", "2019-05-15")],
                "suporte": ("SC", "b"), "resistencia": ("AR", "a")},
    "NVDA": {"titulo": "NVIDIA: distribuição com UTAD (2024 e 2025)", "tipo": "distribuição",
             "janela": ("2024-09-15", "2025-04-15"),
             "eventos": [("PSY", "2024-10-15", "a"), ("BC", "2024-11-21", "a"), ("AR", "2024-11-27", "b"),
                         ("ST", "2024-12-05", "a"), ("UTAD", "2025-01-07", "a"), ("SOW", "2025-01-27", "b"),
                         ("LPSY", "2025-02-18", "a"), ("SOW", "2025-02-27", "b")],
             "fases": [("A", "2024-10-15", "2024-12-05"), ("B", "2024-12-06", "2025-01-06"),
                       ("C", "2025-01-07", "2025-01-24"), ("D", "2025-01-27", "2025-02-26"),
                       ("E", "2025-02-27", "2025-04-15")],
             "suporte": ("2024-12-17", "b"), "resistencia": ("BC", "a")},
    "SUZB3.SA": {"titulo": "SUZB3: distribuição com UTAD (2021 e 2022)", "tipo": "distribuição",
                 "janela": ("2021-11-01", "2022-07-29"),
                 "eventos": [("PSY", "2021-12-01", "a"), ("BC", "2022-01-19", "a"), ("AR", "2022-01-26", "b"),
                             ("ST", "2022-02-09", "a"), ("SOW", "2022-02-23", "b"), ("UTAD", "2022-03-18", "a"),
                             ("SOW", "2022-04-01", "b"), ("LPSY", "2022-04-06", "a"), ("SOW", "2022-04-29", "b"),
                             ("LPSY", "2022-05-05", "a")],
                 "fases": [("A", "2021-12-01", "2022-02-09"), ("B", "2022-02-10", "2022-03-17"),
                           ("C", "2022-03-18", "2022-03-24"), ("D", "2022-03-25", "2022-05-05"),
                           ("E", "2022-05-06", "2022-07-29")],
                 "suporte": ("2022-02-23", "b"), "resistencia": ("BC", "a")},
}


def nivel(d, ex, ref):
    nome, lado = ref
    data = next((dt for n, dt, _ in ex["eventos"] if n == nome), nome)
    return d.loc[data, "Low" if lado == "b" else "High"]


TAB_EX = {}
print("\nParte 2. Quatro exemplos reais (rótulos do autor do capítulo; volume relativo = volume / média de 20 dias)")
for t, ex in EXEMPLOS.items():
    d = serie(t, DX).copy()
    d["rvol"] = d["Volume"] / d["Volume"].rolling(20).mean().shift(1)
    sup, res = nivel(d, ex, ex["suporte"]), nivel(d, ex, ex["resistencia"])
    linhas = []
    for nome, data, lado in ex["eventos"]:
        r = d.loc[data]
        linhas.append((nome, pd.Timestamp(data), r["Low"] if lado == "b" else r["High"], r["Close"], r["rvol"]))
    TAB_EX[t] = {"linhas": linhas, "sup": sup, "res": res}
    fim_fase_e = pd.Timestamp(ex["fases"][-1][2])
    ini_e = pd.Timestamp(ex["fases"][-1][1])
    mov = d.loc[ini_e:fim_fase_e, "Close"]
    extremo = mov.max() if ex["tipo"] == "acumulação" else mov.min()
    TAB_EX[t]["fase_e"] = extremo / d.loc[:ini_e, "Close"].iloc[-1] - 1
    print(f"   {ex['titulo']}: faixa de {nr(sup)} a {nr(res)}")
    print("     " + " | ".join(f"{n} {dt:%d/%m/%y} {nr(p)} ({nr(v, 1)}x)" for n, dt, p, _, v in linhas))
    chave = next(dt for n, dt, _ in ex["eventos"] if n in ("Spring", "Teste", "UTAD"))   # o evento da fase C
    janela = d.loc[chave:, "Close"].iloc[:121]
    ext = janela.max() if ex["tipo"] == "acumulação" else janela.min()
    TAB_EX[t]["chave"] = (pd.Timestamp(chave), d.loc[chave, "Close"], ext / d.loc[chave, "Close"] - 1, janela.idxmax()
                          if ex["tipo"] == "acumulação" else janela.idxmin())
    print(f"     fase E até {fim_fase_e:%d/%m/%Y}: {ps(TAB_EX[t]['fase_e'])} desde o início da fase; do fechamento"
          f" do evento da fase C ({pd.Timestamp(chave):%d/%m/%Y}) ao extremo dos 120 pregões seguintes:"
          f" {ps(TAB_EX[t]['chave'][2])} ({TAB_EX[t]['chave'][3]:%d/%m/%Y})")

In [ ]:
import matplotlib.pyplot as plt
for t, ex in EXEMPLOS.items():
    d = serie(t, DX).loc[ex["janela"][0]:ex["janela"][1]]
    ax = d["Close"].plot(figsize=(9, 3.5), color="black", title=ex["titulo"])
    for nome, data, lado in ex["eventos"]:
        y = d.loc[data, "Low" if lado == "b" else "High"]
        ax.annotate(nome, (pd.Timestamp(data), y), fontsize=8, ha="center")
    plt.show()

## Parte 3: molas e falsos rompimentos

In [ ]:
FAIXA, H = 40, (10, 20)
linhas = []
for t in TICKERS:
    d = serie(t)
    if len(d) < 500:
        continue
    h, l, c, v = (d[k] for k in ["High", "Low", "Close", "Volume"])
    mn, mx = l.rolling(FAIXA).min().shift(1), h.rolling(FAIXA).max().shift(1)   # faixa dos 40 pregões anteriores
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(20).mean()
    rvol = v / v.rolling(20).mean().shift(1)
    lateral = (mx - mn) <= 6 * atr.rolling(100).mean()                         # faixa estreita para a volatilidade
    fut = {k: c.shift(-k) / c - 1 for k in H}
    sinais = {"mola": lateral & (l < mn) & (c > mn), "quebra para baixo (fechou fora)": lateral & (c < mn),
              "falso rompimento para cima": lateral & (h > mx) & (c < mx),
              "rompimento para cima (fechou fora)": lateral & (c > mx), "qualquer dia em faixa": lateral}
    for nome, s in sinais.items():
        idx = s[s].index
        if nome != "qualquer dia em faixa":                                     # um sinal a cada 10 pregões
            ultimo, filtrado = None, []
            for x in idx:
                if ultimo is None or (d.index.get_loc(x) - d.index.get_loc(ultimo)) >= 10:
                    filtrado.append(x)
                    ultimo = x
            idx = pd.Index(filtrado)
        for x in idx:
            if nome == "mola":
                p = d.index.get_loc(x)
                for q in range(p + 1, min(p + 6, len(d))):                      # teste: mínima acima, volume menor
                    if l.iloc[q] > l.iloc[p] and rvol.iloc[q] < rvol.iloc[p] and c.iloc[q] > mn.iloc[p]:
                        y = d.index[q]
                        linhas.append({"t": t, "sinal": "mola com teste (entrada no teste)", "rvol": rvol.iloc[q],
                                       "br": t.endswith(".SA"), **{f"r{k}": fut[k].get(y) for k in H}})
                        break
                topo, piso = mx.iloc[p], l.iloc[p]                              # SOS: fecha acima do topo da faixa
                for q in range(p + 1, min(p + 31, len(d))):
                    if l.iloc[q] < piso:
                        break
                    if c.iloc[q] > topo:
                        y = d.index[q]
                        linhas.append({"t": t, "sinal": "mola + SOS (entrada no rompimento)", "rvol": rvol.iloc[q],
                                       "br": t.endswith(".SA"), **{f"r{k}": fut[k].get(y) for k in H}})
                        for z in range(q + 1, min(q + 26, len(d))):             # reteste a até 1 ATR do topo
                            if c.iloc[z] < topo - atr.iloc[z]:
                                break
                            if l.iloc[z] <= topo + atr.iloc[z]:
                                w = d.index[z]
                                linhas.append({"t": t, "sinal": "mola + SOS + reteste (entrada no reteste)",
                                               "rvol": rvol.iloc[z], "br": t.endswith(".SA"),
                                               **{f"r{k}": fut[k].get(w) for k in H}})
                                break
                        break
            linhas.append({"t": t, "sinal": nome, "rvol": rvol.get(x), "br": t.endswith(".SA"),
                           **{f"r{k}": fut[k].get(x) for k in H}})

TAB = pd.DataFrame(linhas).dropna(subset=["r10", "r20"])
ORDEM = ["mola", "mola com teste (entrada no teste)", "mola + SOS (entrada no rompimento)",
         "mola + SOS + reteste (entrada no reteste)", "quebra para baixo (fechou fora)", "falso rompimento para cima",
         "rompimento para cima (fechou fora)", "qualquer dia em faixa"]
TAB_MOLAS = []
print("\nParte 3. Molas e falsos rompimentos (40 ações, desde 2005)")
print("   sinal                                         casos | 20 pregões: média (positivos)")
for nome in ORDEM:
    x = TAB[TAB["sinal"] == nome]
    TAB_MOLAS.append((nome, len(x), x["r20"].mean(), (x["r20"] > 0).mean()))
    print(f"   {nome:<44} {len(x):>6} | {ps(x['r20'].mean())} ({pc((x['r20'] > 0).mean())})")
M = TAB[TAB["sinal"] == "mola"]
TAB_VOL = []
print("   Mola por volume relativo:")
for nome, cond in [("volume baixo (abaixo da média)", M["rvol"] < 1), ("volume normal (1 a 1,5x)", M["rvol"].between(1, 1.5)),
                   ("volume alto (acima de 1,5x)", M["rvol"] > 1.5)]:
    x = M[cond]
    TAB_VOL.append((nome, len(x), x["r20"].mean(), (x["r20"] > 0).mean()))
    print(f"     {nome:<34} {len(x):>5} | {ps(x['r20'].mean())} ({pc((x['r20'] > 0).mean())})")

## Parte 4: esforço sem resultado

In [ ]:
esforco = []
for t in TICKERS:                                                               # as mesmas 40 ações da Parte 3
    d = serie(t)
    if len(d) < 500:
        continue
    h, l, c, v = (d[k] for k in ["High", "Low", "Close", "Volume"])
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(20).mean()
    rvol = v / v.rolling(20).mean().shift(1)
    fut = {k: c.shift(-k) / c - 1 for k in H}
    # esforço sem resultado (volume 2x a média e barra menor que 60% do ATR)
    perto_topo = c >= 0.98 * h.rolling(20).max()
    perto_fundo = c <= 1.02 * l.rolling(20).min()
    esf = (rvol >= 2) & ((h - l) <= 0.6 * atr)
    for nome, s in [("esforço sem resultado no topo de 20 dias", esf & perto_topo),
                    ("esforço sem resultado no fundo de 20 dias", esf & perto_fundo), ("qualquer dia", c.notna())]:
        esforco.append(pd.DataFrame({"sinal": nome, "r10": fut[10][s], "r20": fut[20][s]}))
E = pd.concat(esforco).dropna()
TAB_ESF = []
print("\nParte 4. Esforço sem resultado (volume 2x a média e barra menor que 60% do ATR)")
for nome in ["esforço sem resultado no topo de 20 dias", "esforço sem resultado no fundo de 20 dias", "qualquer dia"]:
    x = E[E["sinal"] == nome]
    TAB_ESF.append((nome, len(x), x["r20"].mean(), (x["r20"] > 0).mean()))
    print(f"   {nome:<42} {len(x):>7} | 20 pregões {ps(x['r20'].mean())} ({pc((x['r20'] > 0).mean())})")

## Parte 5: causa e efeito

In [ ]:
caixas = []
for t in TICKERS:                                                               # as mesmas 40 ações da Parte 3
    d = serie(t)
    if len(d) < 500:
        continue
    h, l, c = (d[k] for k in ["High", "Low", "Close"])
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(20).mean()
    # causa e efeito (caixas de 30 pregões com amplitude de até 4 ATRs médios)
    rng = h.rolling(30).max() - l.rolling(30).min()
    cond = (rng <= 4 * atr.rolling(100).mean()).fillna(False).to_numpy()
    i, n = 0, len(d)
    while i < n:
        if not cond[i]:
            i += 1
            continue
        j = i
        while j + 1 < n and cond[j + 1]:
            j += 1
        ini = max(i - 29, 0)
        topo, fundo = h.iloc[ini:j + 1].max(), l.iloc[ini:j + 1].min()
        altura, dur = topo - fundo, j - ini + 1
        depois = c.iloc[j + 1:j + 61]
        saida = depois[(depois > topo) | (depois < fundo)]
        if len(saida) and altura > 0:
            k = d.index.get_loc(saida.index[0])
            jan = d.iloc[k:k + 121]
            if len(jan) == 121:
                alta = saida.iloc[0] > topo
                efeito = (jan["High"].max() - topo) / altura if alta else (fundo - jan["Low"].min()) / altura
                caixas.append({"t": t, "dur": dur, "efeito": efeito, "alta": alta,
                               "efeito_atr": efeito * altura / atr.rolling(100).mean().iloc[j]})
        i = j + 1
CX = pd.DataFrame(caixas)
CX["grupo"] = pd.qcut(CX["dur"], 4, labels=["curtas", "médias", "longas", "muito longas"])
TAB_CX = []
print(f"\nParte 5. Causa e efeito: {len(CX)} faixas laterais com rompimento (avanço em 120 pregões, em alturas da faixa)")
for g, x in CX.groupby("grupo", observed=True):
    TAB_CX.append((g, x["dur"].min(), x["dur"].max(), len(x), x["efeito"].median(), x["efeito_atr"].median(),
                   (x["efeito"] >= 1).mean()))
    print(f"   faixas {g:<13} ({x['dur'].min():>3} a {x['dur'].max():>3} pregões, {len(x):>4} casos): mediana"
          f" {nr(x['efeito'].median())} alturas ({nr(x['efeito_atr'].median(), 1)} ATRs) | chegou a 1 altura em"
          f" {pc((x['efeito'] >= 1).mean())}")
RHO = (CX["dur"].corr(CX["efeito"], method="spearman"), CX["dur"].corr(CX["efeito_atr"], method="spearman"))
print(f"   correlação de postos (duração x avanço): {nr(RHO[0])} em alturas, {nr(RHO[1])} em ATRs;"
      f" rompimentos para cima: {pc(CX['alta'].mean())}")

## Parte 6: o teste do tempo do S&P 500

In [ ]:
SPX = serie("^GSPC")["Close"]
FUNDO = SPX.loc["2009"].idxmin()
ALVOS = {}
print(f"\nParte 6. S&P 500: fundo de {nr(SPX[FUNDO])} pontos em {FUNDO:%d/%m/%Y}")
for alvo in (1270, 1930):
    dia = SPX.loc[FUNDO:][SPX.loc[FUNDO:] >= alvo].index[0]
    ALVOS[alvo] = dia
    print(f"   alvo de {nr(alvo, 0)}: primeiro fechamento acima em {dia:%d/%m/%Y}, {(dia - FUNDO).days} dias depois")

## Peça à IA

O Colab tem um assistente de IA. Abra o assistente, cole um destes pedidos e confira a resposta rodando a célula indicada.

1. **Explicar:** "Explique, linha por linha, o começo da célula da Parte 3, até o dicionário `sinais`. Mostre por que `mn`, `mx` e a média do volume em `rvol` usam `shift(1)` e como `lateral & (l < mn) & (c > mn)` traduz a mola: furar a mínima da faixa e fechar de volta dentro dela." Confira rodando a célula da Parte 3, que imprime a tabela das molas (no livro, 1.556 molas, com +0,9% em 20 pregões).
2. **Adaptar:** "Numa célula nova, depois da Parte 3, baixe [ATIVO] com `yf.download(['[TICKER]'], start='2005-01-01', end=FIM, auto_adjust=True, group_by='ticker')`, limpe com `serie('[TICKER]', novo)` e aplique as mesmas linhas da Parte 3 até `sinais`. Liste as datas das molas, com o `rvol` de cada uma. Não mude `FAIXA`, o `shift(1)`, o limite de 6 ATRs nem o espaçamento de um sinal a cada 10 pregões." Antes de olhar o ativo novo, confira se a Parte 3 continua com os números do livro (1.556 molas; mola + SOS com +1,2% e 60% de casos positivos, em 175 casos).

Mais pedidos deste capítulo (entender e aplicar ao seu gráfico) estão na página do capítulo, no hub do livro.

## Para ir além
1. Na Parte 2, escolha uma faixa de uma ação que você acompanha e rotule os eventos você mesmo.
2. Na Parte 3, troque a faixa de 40 para 60 pregões. A mola com teste melhora?
3. Na Parte 4, exija volume de 3 vezes a média. O esforço sem resultado no topo muda de sinal?
4. Na Parte 5, meça o efeito em 60 pregões em vez de 120. Faixas longas rendem mais?

**Governança:** os resultados deste notebook valem só para os dados e as datas usadas. Mudar regras até o resultado ficar bonito é sobreajuste (*overfitting*): a regra decora o passado e falha no futuro.

━━━━━━━━━━━━━━━━━━━━━━━

⚠️ AVISO LEGAL (DISCLAIMER):

━━━━━━━━━━━━━━━━━━━━━━━

Todo o conteúdo deste livro  é estritamente educacional e informativo. 

Não constitui, em nenhuma hipótese, recomendação de compra ou venda de ativos financeiros. 

Testes passados não são garantias de resultados futuros.

Investimentos envolvem riscos, incluindo a perda total do capital investido. 

Consulte sempre um profissional habilitado antes de tomar qualquer decisão de investimento.